In [1]:
#用于生成unet网络pytorch版本
def Auto_Unet_pytorch(vy,vx,downnum,covnum,test_size=0.2,valid_size=0.1,k_fold=None,task_mode='regression',if_best_mode='no',modelpath=None,ifrandom_split='yes',baselayer=64,cov_kernelsize=3,cov_strides=1,pool_method='maxpooling',pool_kernel_size=2,pool_strides=2,ifnormalization='no',normalization_method=None,activate='tanh',if_last_act='no',if_print_model='yes',loss_function='default',optimizer='SGD',metrics='default',learning_rate=0.01,epochs=2000,batch_size=20,if_early_stopping=None,ifheatmap='no',ifweight='yes',ifmute='no',ifsave='no',savepath=None,device='cpu'):
    from torch.nn import Module,BatchNorm1d,BatchNorm2d,LayerNorm,Conv2d,MaxPool2d,AvgPool2d,Dropout,LeakyReLU,ReLU,PReLU,Sigmoid,Tanh,ELU,Softmax,Linear,Flatten,ZeroPad2d,Upsample
    from torch.optim import Adam,NAdam,SGD
    import torch
    from torch import nn
    import torchmetrics
    import math
    from sklearn.model_selection import train_test_split
    from sklearn.model_selection import KFold
    import numpy as np
    from scipy.stats import pearsonr
    import os
    from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score
    import sklearn
    import copy
    import shap
    import datetime
    import warnings
    from tqdm import tqdm

    # 忽略特定警告
    warnings.filterwarnings("ignore")
    
    # ============================================================
    # Device selection
    # 支持：
    #   device='cpu'
    #   device='gpu' 或 device='cuda'  -> cuda:0
    #   device='cuda:0'、'cuda:1'、'cuda:2'...
    #   device=0、1、2...
    #   device=torch.device(...)
    # ============================================================
    if isinstance(device, torch.device):
        devices = device

    elif isinstance(device, (int, np.integer)):
        devices = torch.device(f'cuda:{int(device)}')

    elif isinstance(device, str):
        device_lower = device.strip().lower()

        if device_lower == 'cpu':
            devices = torch.device('cpu')

        elif device_lower in ('gpu', 'cuda'):
            devices = torch.device('cuda:0')

        elif device_lower.startswith('cuda:'):
            gpu_index_text = device_lower.split(':', 1)[1]

            if not gpu_index_text.isdigit():
                raise ValueError(
                    f"device={device!r} 格式错误。"
                    "GPU设备应写成 'cuda:0'、'cuda:1' 等。"
                )

            devices = torch.device(
                f"cuda:{int(gpu_index_text)}"
            )

        else:
            raise ValueError(
                f"不支持的 device={device!r}。"
                "请使用 'cpu'、'gpu'、'cuda'、"
                "'cuda:0'、'cuda:1' 等，"
                "或者直接传入整数GPU编号。"
            )

    else:
        raise TypeError(
            "device 必须是 str、int、numpy整数或 torch.device，"
            f"当前类型为 {type(device).__name__}。"
        )

    # 检查 CUDA 可用性和设备编号
    if devices.type == 'cuda':
        if not torch.cuda.is_available():
            raise RuntimeError(
                f"指定了 {devices}，但当前 PyTorch 无法使用 CUDA。\n"
                f"PyTorch版本：{torch.__version__}\n"
                f"PyTorch内置CUDA版本：{torch.version.cuda}"
            )

        gpu_index = (
            0 if devices.index is None
            else int(devices.index)
        )

        visible_gpu_count = torch.cuda.device_count()

        if gpu_index < 0 or gpu_index >= visible_gpu_count:
            raise RuntimeError(
                f"指定了 cuda:{gpu_index}，但当前进程只识别到 "
                f"{visible_gpu_count} 张GPU。\n"
                f"CUDA_VISIBLE_DEVICES="
                f"{os.environ.get('CUDA_VISIBLE_DEVICES')}\n"
                "注意：设置 CUDA_VISIBLE_DEVICES 后，"
                "PyTorch会对当前进程可见的GPU重新从 cuda:0 编号。"
            )

        devices = torch.device(f'cuda:{gpu_index}')
        torch.cuda.set_device(devices)

    print('=' * 70)
    print('实际使用设备：', devices)

    if devices.type == 'cuda':
        print(
            'GPU名称：',
            torch.cuda.get_device_name(devices)
        )
        print(
            '当前进程可见GPU数量：',
            torch.cuda.device_count()
        )
        print(
            'CUDA_VISIBLE_DEVICES：',
            os.environ.get('CUDA_VISIBLE_DEVICES')
        )

    print('=' * 70)
    if task_mode=='binary_classify' or task_mode=='multi_classify':
        def get_class_weights_from_labels(vy, device=None, ignore_index=None, num_classes=None):
            y = torch.as_tensor(vy, device=device)
        
            # 如果你标签存成 (...,1)，内部 squeeze 掉这个 1（不改原 vy）
            if y.ndim >= 1 and y.shape[-1] == 1:
                y = y.squeeze(-1)
        
            y = y.reshape(-1)
        
            if ignore_index is not None:
                y = y[y != ignore_index]
        
            if y.is_floating_point():
                y = y[~torch.isnan(y)]
        
            y = y.long()
        
            if num_classes is None:
                num_classes = int(y.max().item()) + 1
        
            cnt = torch.bincount(y, minlength=num_classes).float()
            w = cnt.sum() / (num_classes * cnt.clamp_min(1.0))  # N/(C*count_c)
            return w  # shape: (C,)
        class_weights = get_class_weights_from_labels(vy, device=devices)
    if task_mode=='regression':
        if type(loss_function) is not str:
            loss=loss_function
        else:
            if loss_function=='default' or loss_function=='MSELoss':
                loss=torch.nn.MSELoss()
            elif loss_function=='L1Loss':
                loss=torch.nn.L1Loss
            elif loss_function=='PoissonNLLLoss':
                loss=torch.nn.PoissonNLLLoss()
            elif loss_function=='GaussianNLLLoss':
                loss=torch.nn.GaussianNLLLoss()
            elif loss_function=='KLDivLoss':
                loss=torch.nn.KLDivLoss()
            elif loss_function=='HuberLoss':
                loss=torch.nn.HuberLoss()
            elif loss_function=='SmoothL1Loss':
                loss=torch.nn.SmoothL1Loss()
            elif loss_function=='Pearsonr':
                class loss_pearsonr(nn.Module):
                    def __init__(self):
                        super().__init__()
    
                    def forward(self, y, x):
                        y_true_mean=torch.nanmean(y,dim=0,keepdim=True)
                        y_pred_mean=torch.nanmean(x,dim=0,keepdim=True)
                        cov=torch.nansum((y-y_true_mean)*(x-y_pred_mean),dim=0,keepdim=True)
                        y_true_v=torch.nansum(torch.square((y-y_true_mean)),dim=0,keepdim=True)
                        y_pred_v=torch.nansum(torch.square((x-y_pred_mean)),dim=0,keepdim=True)
                        y_true_v=torch.sqrt(y_true_v)
                        y_pred_v=torch.sqrt(y_pred_v)
                        pearson=cov/(y_true_v*y_pred_v)
                        return (1-pearson)**1.5
                loss=loss_pearsonr()
        if type(metrics) is not str:
            metric=metrics
        else:
            if metrics=='default' or metrics=='MSELoss':
                metric=torch.nn.MSELoss()
            elif metrics=='L1Loss':
                metric=torch.nn.L1Loss
            elif metrics=='PoissonNLLLoss':
                metric=torch.nn.PoissonNLLLoss()
            elif metrics=='GaussianNLLLoss':
                metric=torch.nn.GaussianNLLLoss()
            elif metrics=='KLDivLoss':
                metric=torch.nn.KLDivLoss()
            elif metrics=='HuberLoss':
                metric=torch.nn.HuberLoss()
            elif metrics=='SmoothL1Loss':
                metric=torch.nn.SmoothL1Loss()
            elif metrics=='Pearsonr':
                class metric_pearsonr(nn.Module):
                    def __init__(self):
                        super().__init__()
    
                    def forward(self, y, x):
                        y_true_mean=torch.nanmean(y,dim=0,keepdim=True)
                        y_pred_mean=torch.nanmean(x,dim=0,keepdim=True)
                        cov=torch.nansum((y-y_true_mean)*(x-y_pred_mean),dim=0,keepdim=True)
                        y_true_v=torch.nansum(torch.square((y-y_true_mean)),dim=0,keepdim=True)
                        y_pred_v=torch.nansum(torch.square((x-y_pred_mean)),dim=0,keepdim=True)
                        y_true_v=torch.sqrt(y_true_v)
                        y_pred_v=torch.sqrt(y_pred_v)
                        pearson=cov/(y_true_v*y_pred_v)
                        return (1-pearson)**1.5
                metric=metric_pearsonr()
    elif task_mode=='binary_classify':
        if type(loss_function) is not str:
            loss=loss_function
        else:
            if loss_function=='default' or loss_function=='BCELoss':
                w0, w1 = class_weights[0], class_weights[1]
                weight_map = torch.where(y_batch > 0.5, w1, w0)
                loss=torch.nn.BCELoss(weight=weight_map)
            elif loss_function=='BCEWithLogitsLoss':
                pos_w = (class_weights[1] / class_weights[0]).to(devices).view(1)
                loss=torch.nn.BCEWithLogitsLoss(pos_weight=pos_w)
            elif loss_function=='SoftMarginLoss':
                loss=torch.nn.SoftMarginLoss()
            elif loss_function=='MultiLabelSoftMarginLoss':
                loss=torch.nn.MultiLabelSoftMarginLoss()
        if type(metrics) is not str:
            metric=metrics
        else:
            if metrics=='default' or metrics=='f1':
                metric=torchmetrics.F1Score(task="binary").to(devices)
            elif metrics=='accuracy':
                metric=torchmetrics.Accuracy(task="binary").to(devices)
            elif metrics=='precision':
                metric=torchmetrics.Precision(task="binary").to(devices)
            elif metrics=='recall':
                metric=torchmetrics.Recall(task="binary").to(devices)
            elif metrics=='BCELoss':
                metric=torch.nn.BCELoss()
            elif metrics=='BCEWithLogitsLoss':
                metric=torch.nn.BCEWithLogitsLoss()
            elif metrics=='SoftMarginLoss':
                metric=torch.nn.SoftMarginLoss()
            elif metrics=='MultiLabelSoftMarginLoss':
                metric=torch.nn.MultiLabelSoftMarginLoss()
    elif task_mode=='multi_classify':
        if type(loss_function) is not str:
            loss=loss_function
        else:
            if loss_function=='default' or loss_function=='CrossEntropyLoss':
                loss=torch.nn.CrossEntropyLoss(weight=class_weights)
            elif loss_function=='NLLLoss':
                loss=torch.nn.NLLLoss(weight=class_weights)
            elif loss_function=='TripletMarginLoss':
                loss=torch.nn.TripletMarginLoss()
            elif loss_function=='KLDivergence':
                loss=torch.nn.KLDivergence()
            elif loss_function=='HingeEmbeddingLoss':
                loss=torch.nn.HingeEmbeddingLoss()
            elif loss_function=='MultiLabelMarginLoss':
                loss=torch.nn.MultiLabelMarginLoss()
            elif loss_function=='TripletMarginWithDistanceLoss':
                loss=torch.nn.TripletMarginWithDistanceLoss()
        if type(metrics) is not str:
            metric=metrics
        else:
            if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                metric=torchmetrics.Accuracy(task="multiclass",num_classes = int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='recall' :
                metric=torchmetrics.Recall(task="multiclass", num_classes=int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='precision' :
                metric=torchmetrics.Precision(task="multiclass", num_classes=int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='f1' :
                metric=torchmetrics.F1Score(task="multiclass", num_classes=int(np.max(vy))+1, average="macro").to(devices)
            elif metrics=='CrossEntropyLoss':
                metric=torch.nn.CrossEntropyLoss()
            elif metrics=='NLLLoss':
                metric=torch.nn.NLLLoss()
            elif metrics=='TripletMarginLoss':
                metric=torch.nn.TripletMarginLoss()
            elif metrics=='KLDivergence':
                metric=torch.nn.KLDivergence()
            elif metrics=='HingeEmbeddingLoss':
                metric=torch.nn.HingeEmbeddingLoss()
            elif metrics=='MultiLabelMarginLoss':
                metric=torch.nn.MultiLabelMarginLoss()
            elif metrics=='TripletMarginWithDistanceLoss':
                metric=torch.nn.TripletMarginWithDistanceLoss()
    weights=0
    heatmap = 0
    model=0
    class ShapWrapper(Module):
        def __init__(self, original_model, x_sample_shape, output_channel_idx=None):
            super().__init__()
            self.original_model = original_model
            self.x_sample_shape = x_sample_shape
            self.x_flat_size = np.prod(x_sample_shape)
            self.output_channel_idx = output_channel_idx
        def forward(self, x_flat_tensor):
            if x_flat_tensor.dim() == 1:
                x_flat_tensor = x_flat_tensor.unsqueeze(0)
            x_reshaped = x_flat_tensor.reshape(-1, *self.x_sample_shape)
            model_output = self.original_model(x_reshaped)
            if model_output.is_floating_point() is False:
                model_output = model_output.float()
            if self.output_channel_idx is not None:
                channel_output = model_output[:, self.output_channel_idx, :, :]
                scalar_per_sample = torch.mean(channel_output.view(channel_output.size(0), -1), dim=1)
            else:
                scalar_per_sample = torch.mean(model_output.view(model_output.size(0), -1), dim=1)
            return scalar_per_sample.unsqueeze(1)
    class EarlyStopping:
        def __init__(self, patience, delta=0):
            self.patience = patience
            self.counter = 0
            self.best_score = None
            self.early_stop = False
            self.val_loss_min = np.Inf
            self.delta = delta
            self.best_model_state_dict = None
    
        def __call__(self, val_loss, model):
            # 1. NaN 保护：一旦验证 loss 是 NaN，就停止训练，后面外层用 load_best_checkpoint 恢复
            if np.isnan(val_loss):
                print("Validation loss is NaN. Stopping early and will restore best model.")
                self.early_stop = True
                return
    
            score = -val_loss
    
            # 2. 正常 early stopping 逻辑
            if self.best_score is None:
                self.best_score = score
                self.save_checkpoint(val_loss, model)
            elif score < self.best_score + self.delta:
                # 没有提升
                self.counter += 1
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                # 有提升，更新 best
                self.best_score = score
                self.save_checkpoint(val_loss, model)
                self.counter = 0
    
        def save_checkpoint(self, val_loss, model):
            # 这里仍然是 clone 当前最优模型
            self.best_model_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
            self.val_loss_min = val_loss
    
        def load_best_checkpoint(self, model):
            # 新增：一行恢复 best 模型
            if self.best_model_state_dict is not None:
                model.load_state_dict(self.best_model_state_dict)
            else:
                print("Warning: load_best_checkpoint called but no best_model_state_dict was saved.")
    vx=vx.transpose(0,3,1,2)
    if vy.ndim==3:
        vy=vy.reshape(vy.shape[0],vy.shape[1],vy.shape[2],1)
    vy=vy.transpose(0,3,1,2)
    model=None
    predicty=None
    testy=None
    r=None
    p=None
    
    if ifrandom_split=='yes':
        trainy,testy,trainx,testx = train_test_split(vy,vx,test_size=test_size,random_state=25)
    elif ifrandom_split=='no':
        index=int((1-test_size)*vy.shape[0])
        trainy=vy[:index,:,:,:]
        testy=vy[index:,:,:,:]
        trainx=vx[:index,:,:,:]
        testx=vx[index:,:,:,:]
    elif ifrandom_split=='all_train' or ifrandom_split=='all_test':
        trainy=vy
        testy=vy
        trainx=vx
        testx=vx
    class Model(nn.Module):
        def __init__(self,trainx,trainy,downnum,covnum,baselayer,cov_kernelsize,cov_strides,pool_method,pool_kernel_size,pool_strides,ifnormalization,normalization_method,activate):
            super(Model,self).__init__()
            exec('from torch.nn import Module,BatchNorm1d,BatchNorm2d,LayerNorm,Conv2d,MaxPool2d,AvgPool2d,Dropout,LeakyReLU,ReLU,PReLU,Sigmoid,Tanh,ELU,Softmax,Linear,Flatten,ZeroPad2d,Upsample', globals(), self.__dict__)
            exec('import numpy as np', globals(), self.__dict__)
            exec('import math', globals(), self.__dict__)
            exec('import torch', globals(), self.__dict__)
            self.downnum=downnum
            self.covnum=covnum
            self.baselayer=baselayer
            self.cov_kernelsize=cov_kernelsize
            self.cov_strides=cov_strides
            self.pool_method=pool_method
            self.pool_kernel_size=pool_kernel_size
            self.pool_strides=pool_strides
            self.ifnormalization=ifnormalization
            self.normalization_method=normalization_method
            self.activate=activate
            self.trainx=trainx
            self.trainy=trainy
            self.hight0=self.trainx.shape[2]
            self.weight0=self.trainx.shape[3]
            self.upsize=pool_strides
            self.__dict__['self']=self
            self.k=0
            self.total_covnum=0
            
            W=trainx.shape[2]
            H=trainx.shape[3]
            for i in range(int(self.downnum-1)):
                W=math.floor(((W-self.pool_kernel_size)/self.pool_strides)+1)
                H=math.floor(((H-self.pool_kernel_size)/self.pool_strides)+1)
            if W<1 or H<1:
                print('下采样次数过多,导致最下层图形长度/宽度小于1!')
                return model,predicty,testy,r,p
            if self.pool_strides!=self.pool_kernel_size:
                print('池化步长和池化核不等,将导致下采样前后通道数相除不为整数！')
                return model,predicty,testy,r,p
            for i in range(int(self.downnum)):
                for j in range(int(self.covnum)):
                    if self.total_covnum==0:
                        exec('self.pad'+str(i+1)+'_'+str(j+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.trainx.shape[3]-(self.trainx.shape[3]/self.cov_strides)))/2.0),math.ceil(math.floor((self.trainx.shape[3]-(self.trainx.shape[3]/self.cov_strides)))/2.0),math.floor(math.floor((self.trainx.shape[2]-(self.trainx.shape[2]/self.cov_strides)))/2.0),math.ceil(math.floor((self.trainx.shape[2]-(self.trainx.shape[2]/self.cov_strides)))/2.0)))', globals(), self.__dict__)
                        exec('self.conv'+str(i+1)+'_'+str(j+1)+'=Conv2d(self.trainx.shape[1],self.baselayer,(self.cov_kernelsize,self.cov_kernelsize),stride=self.cov_strides,padding="same")', globals(), self.__dict__)
                        if self.activate=='elu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ELU()', globals(), self.__dict__)
                        elif self.activate=='leakyrelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=LeakyReLU()', globals(), self.__dict__)
                        elif self.activate=='prelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=PReLU()', globals(), self.__dict__)
                        elif self.activate=='relu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ReLU()', globals(), self.__dict__)
                        elif self.activate=='sigmoid':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Sigmoid()', globals(), self.__dict__)
                        elif self.activate=='tanh':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Tanh()', globals(), self.__dict__)
                        elif self.activate=='softmax':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Softmax()', globals(), self.__dict__)
                        exec('in_channels=self.baselayer', globals(), self.__dict__)
                    else:
                        exec('self.pad'+str(i+1)+'_'+str(j+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.weight'+str(i)+'-(self.weight'+str(i)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.weight'+str(i)+'-(self.weight'+str(i)+'/self.cov_strides)))/2.0),math.floor(math.floor((self.hight'+str(i)+'-(self.hight'+str(i)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.hight'+str(i)+'-(self.hight'+str(i)+'/self.cov_strides)))/2.0)))', globals(), self.__dict__)
                        exec('self.conv'+str(i+1)+'_'+str(j+1)+'=Conv2d(self.in_channels,self.baselayer,(self.cov_kernelsize,self.cov_kernelsize),stride=self.cov_strides,padding="same")', globals(), self.__dict__)
                        if self.activate=='elu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ELU()', globals(), self.__dict__)
                        elif self.activate=='leakyrelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=LeakyReLU()', globals(), self.__dict__)
                        elif self.activate=='prelu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=PReLU()', globals(), self.__dict__)
                        elif self.activate=='relu':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=ReLU()', globals(), self.__dict__)
                        elif self.activate=='sigmoid':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Sigmoid()', globals(), self.__dict__)
                        elif self.activate=='tanh':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Tanh()', globals(), self.__dict__)
                        elif self.activate=='softmax':
                            exec('self.act'+str(i+1)+'_'+str(j+1)+'=Softmax()', globals(), self.__dict__)
                        exec('in_channels=self.baselayer', globals(), self.__dict__)
                    self.total_covnum=self.total_covnum+1
                    if self.ifnormalization=='yes':
                        if self.normalization_method=='batchnormalization':
                            exec('self.norm'+str(i+1)+'_'+str(j+1)+'=BatchNorm2d(in_channels)', globals(), self.__dict__)
                        elif self.normalization_method=='layernormalization':
                            exec('self.norm'+str(i+1)+'_'+str(j+1)+'=LayerNorm(in_channels)', globals(), self.__dict__)
                self.k=self.k+1
                if i!=(self.downnum-1):
                    self.baselayer=self.baselayer*2
                if self.pool_method=='maxpooling' and i!=(self.downnum-1):
                    exec('self.hight'+str(i+1)+'=1+math.floor((self.hight'+str(i)+'-self.pool_kernel_size)/self.pool_strides)', globals(), self.__dict__)
                    exec('self.weight'+str(i+1)+'=1+math.floor((self.weight'+str(i)+'-self.pool_kernel_size)/self.pool_strides)', globals(), self.__dict__)
                    exec('self.pool'+str(i+1)+'=MaxPool2d((self.pool_kernel_size,self.pool_kernel_size),stride=self.pool_strides)', globals(), self.__dict__)
                elif self.pool_method=='avepooling' and i!=(self.downnum-1):
                    exec('self.hight'+str(i+1)+'=1+math.floor((self.hight'+str(i)+'-self.pool_kernel_size)/self.pool_strides)', globals(), self.__dict__)
                    exec('self.weight'+str(i+1)+'=1+math.floor((self.weight'+str(i)+'-self.pool_kernel_size)/self.pool_strides)', globals(), self.__dict__)
                    exec('self.pool'+str(i+1)+'=AvgPool2d((self.pool_kernel_size,self.pool_kernel_size),stride=self.pool_strides)', globals(), self.__dict__)
            self.l=self.k-1
            self.k=self.k+1    
            for m in range(self.downnum,2*self.downnum-1):
                self.baselayer=int(self.baselayer/2.0)
                if m==self.downnum:
                    exec('self.pad'+str(m+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.weight'+str(m-1)+'-(self.weight'+str(m-1)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.weight'+str(m-1)+'-(self.weight'+str(m-1)+'/self.cov_strides)))/2.0),math.floor(math.floor((self.hight'+str(m-1)+'-(self.hight'+str(m-1)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.hight'+str(m-1)+'-(self.hight'+str(m-1)+'/self.cov_strides)))/2.0)))', globals(), self.__dict__)
                else:
                    exec('self.pad'+str(m+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.weight'+str(m)+'-(self.weight'+str(m)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.weight'+str(m)+'-(self.weight'+str(m)+'/self.cov_strides)))/2.0),math.floor(math.floor((self.hight'+str(m)+'-(self.hight'+str(m)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.hight'+str(m)+'-(self.hight'+str(m)+'/self.cov_strides)))/2.0)))', globals(), self.__dict__)
                exec('self.conv'+str(m+1)+'=Conv2d(self.in_channels,self.baselayer,(self.cov_kernelsize,self.cov_kernelsize),stride=self.cov_strides,padding="same")', globals(), self.__dict__)
                if self.activate=='elu':
                    exec('self.act'+str(m+1)+'=ELU()', globals(), self.__dict__)
                elif self.activate=='leakyrelu':
                    exec('self.act'+str(m+1)+'=LeakyReLU()', globals(), self.__dict__)
                elif self.activate=='prelu':
                    exec('self.act'+str(m+1)+'=PReLU()', globals(), self.__dict__)
                elif self.activate=='relu':
                    exec('self.act'+str(m+1)+'=ReLU()', globals(), self.__dict__)
                elif self.activate=='sigmoid':
                    exec('self.act'+str(m+1)+'=Sigmoid()', globals(), self.__dict__)
                elif self.activate=='tanh':
                    exec('self.act'+str(m+1)+'=Tanh()', globals(), self.__dict__)
                elif self.activate=='softmax':
                    exec('self.act'+str(m+1)+'=Softmax()', globals(), self.__dict__)
                exec('in_channels=self.baselayer', globals(), self.__dict__)
                if self.ifnormalization=='yes':
                    if self.normalization_method=='batchnormalization':
                        exec('self.norm'+str(m+1)+'=BatchNorm2d(in_channels)', globals(), self.__dict__)
                    elif self.normalization_method=='layernormalization':
                        exec('self.norm'+str(m+1)+'=LayerNorm(in_channels)', globals(), self.__dict__)
                exec('self.hight'+str(m+1)+'=self.hight'+str(self.l)+'*self.upsize', globals(), self.__dict__)
                exec('self.weight'+str(m+1)+'=self.weight'+str(self.l)+'*self.upsize', globals(), self.__dict__)
                exec('self.conc'+str(m+1)+'=torch.concatenate', globals(), self.__dict__)   
                exec('self.uppad'+str(m+1)+'=ZeroPad2d(padding=(math.floor((self.weight'+str(self.l-1)+'-self.weight'+str(self.k)+')/2.0),math.ceil((self.weight'+str(self.l-1)+'-self.weight'+str(self.k)+')/2.0),math.floor((self.hight'+str(self.l-1)+'-self.hight'+str(self.k)+')/2.0),math.ceil((self.hight'+str(self.l-1)+'-self.hight'+str(self.k)+')/2.0)))', globals(), self.__dict__)
                exec('self.up'+str(m+1)+'=Upsample(scale_factor=self.upsize, mode="bilinear")', globals(), self.__dict__)
                exec('in_channels=in_channels*2', globals(), self.__dict__)
                for n in range(int(self.covnum)):
                    exec('self.pad'+str(m+1)+'_'+str(n+1)+'=ZeroPad2d(padding=(math.floor(math.floor((self.weight'+str(m+1)+'-(self.weight'+str(m+1)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.weight'+str(m+1)+'-(self.weight'+str(m+1)+'/self.cov_strides)))/2.0),math.floor(math.floor((self.hight'+str(m+1)+'-(self.hight'+str(m+1)+'/self.cov_strides)))/2.0),math.ceil(math.floor((self.hight'+str(m+1)+'-(self.hight'+str(m+1)+'/self.cov_strides)))/2.0)))', globals(), self.__dict__)
                    exec('self.conv'+str(m+1)+'_'+str(n+1)+'=Conv2d(self.in_channels,self.baselayer,(self.cov_kernelsize,self.cov_kernelsize),stride=self.cov_strides,padding="same")', globals(), self.__dict__)
                    if self.activate=='elu':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=ELU()', globals(), self.__dict__)
                    elif self.activate=='leakyrelu':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=LeakyReLU()', globals(), self.__dict__)
                    elif self.activate=='prelu':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=PReLU()', globals(), self.__dict__)
                    elif self.activate=='relu':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=ReLU()', globals(), self.__dict__)
                    elif self.activate=='sigmoid':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=Sigmoid()', globals(), self.__dict__)
                    elif self.activate=='tanh':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=Tanh()', globals(), self.__dict__)
                    elif self.activate=='softmax':
                        exec('self.act'+str(m+1)+'_'+str(n+1)+'=Softmax()', globals(), self.__dict__)
                    exec('in_channels=self.baselayer', globals(), self.__dict__)
                    if self.ifnormalization=='yes':
                        if self.normalization_method=='batchnormalization':
                            exec('self.norm'+str(m+1)+'_'+str(n+1)+'=BatchNorm2d(in_channels)', globals(), self.__dict__)
                        elif self.normalization_method=='layernormalization':
                            exec('self.norm'+str(m+1)+'_'+str(n+1)+'=LayerNorm(in_channels)', globals(), self.__dict__)
                self.k=self.k+1
                self.l=self.l-1
            if task_mode=='multi_classify':
                exec('self.convlast=Conv2d(self.in_channels,np.max(trainy)+1,(1,1),stride=1,padding="same")', globals(), self.__dict__)
            else:
                exec('self.convlast=Conv2d(self.in_channels,self.trainy.shape[1],(1,1),stride=1,padding="same")', globals(), self.__dict__)
            if if_last_act!='no':
                if if_last_act=='elu':
                    exec('self.actlast=ELU()', globals(), self.__dict__)
                elif if_last_act=='leakyrelu':
                    exec('self.actlast=LeakyReLU()', globals(), self.__dict__)
                elif if_last_act=='prelu':
                    exec('self.actlast=PReLU()', globals(), self.__dict__)
                elif if_last_act=='relu':
                    exec('self.actlast=ReLU()', globals(), self.__dict__)
                elif if_last_act=='sigmoid':
                    exec('self.actlast=Sigmoid()', globals(), self.__dict__)
                elif if_last_act=='tanh':
                    exec('self.actlast=Tanh()', globals(), self.__dict__)
                elif if_last_act=='softmax':
                    exec('self.actlast=Softmax()', globals(), self.__dict__)
        def forward(self, x):
            self.__dict__['x']=x
            self.k=0
            for i in range(int(self.downnum)):
                for j in range(int(self.covnum)):
                    if i==0 and j==0:
                        exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(x)', globals(), self.__dict__)
                    else:
                        if j==0:
                            exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_pool'+str(i)+')', globals(), self.__dict__)
                        else:
                            if self.ifnormalization=='yes':
                                exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_norm'+str(i+1)+'_'+str(j)+')', globals(), self.__dict__)
                            else:
                                exec('model_conv'+str(i+1)+'_'+str(j+1)+'=self.conv'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j)+')', globals(), self.__dict__)
                    exec('model_pad'+str(i+1)+'_'+str(j+1)+'=self.pad'+str(i+1)+'_'+str(j+1)+'(model_conv'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                    exec('model_act'+str(i+1)+'_'+str(j+1)+'=self.act'+str(i+1)+'_'+str(j+1)+'(model_pad'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                    if self.ifnormalization=='yes':
                        exec('model_norm'+str(i+1)+'_'+str(j+1)+'=self.norm'+str(i+1)+'_'+str(j+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)   
                self.k=self.k+1
                if i!=(self.downnum-1):
                    if self.ifnormalization=='yes':
                        exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                    else:
                        exec('model_pool'+str(i+1)+'=self.pool'+str(i+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
            self.l=self.k-1
            self.k=self.k+1
            for m in range(self.downnum,2*self.downnum-1):
                if m==self.downnum:
                    if self.ifnormalization=='yes':
                        exec('model_conv'+str(m+1)+'=self.conv'+str(m+1)+'(model_norm'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                    else:
                        exec('model_conv'+str(m+1)+'=self.conv'+str(m+1)+'(model_act'+str(i+1)+'_'+str(j+1)+')', globals(), self.__dict__)
                else:
                    if self.ifnormalization=='yes':
                        exec('model_conv'+str(m+1)+'=self.conv'+str(m+1)+'(model_norm'+str(m)+'_'+str(n+1)+')', globals(), self.__dict__)
                    else:
                        exec('model_conv'+str(m+1)+'=self.conv'+str(m+1)+'(model_act'+str(m)+'_'+str(n+1)+')', globals(), self.__dict__)
                exec('model_pad'+str(m+1)+'=self.pad'+str(m+1)+'(model_conv'+str(m+1)+')', globals(), self.__dict__)
                exec('model_act'+str(m+1)+'=self.act'+str(m+1)+'(model_pad'+str(m+1)+')', globals(), self.__dict__)
                if self.ifnormalization=='yes':
                    exec('model_norm'+str(m+1)+'=self.norm'+str(m+1)+'(model_act'+str(m+1)+')', globals(), self.__dict__)
                if self.ifnormalization=='yes':
                    exec('self.covup'+str(m+1)+'=model_norm'+str(self.k), globals(), self.__dict__)
                    exec('self.covdown'+str(m+1)+'=model_norm'+str(self.l)+'_'+str(int(self.covnum)), globals(), self.__dict__)
                else:
                    exec('self.covup'+str(m+1)+'=model_conv'+str(self.k), globals(), self.__dict__)
                    exec('self.covdown'+str(m+1)+'=model_conv'+str(self.l)+'_'+str(int(self.covnum)), globals(), self.__dict__)
                exec('model_up'+str(m+1)+'=self.up'+str(m+1)+'(self.covup'+str(m+1)+')', globals(), self.__dict__)
                exec('model_uppad'+str(m+1)+'=self.uppad'+str(m+1)+'(model_up'+str(m+1)+')', globals(), self.__dict__)
                exec('model_conc'+str(m+1)+'=self.conc'+str(m+1)+'((model_uppad'+str(m+1)+',self.covdown'+str(m+1)+'),axis=1)', globals(), self.__dict__)
                for n in range(int(self.covnum)):
                    if n==0:
                        exec('model_conv'+str(m+1)+'_'+str(n+1)+'=self.conv'+str(m+1)+'_'+str(n+1)+'(model_conc'+str(m+1)+')', globals(), self.__dict__)
                    else:
                        if self.ifnormalization=='yes':
                            exec('model_conv'+str(m+1)+'_'+str(n+1)+'=self.conv'+str(m+1)+'_'+str(n+1)+'(model_norm'+str(m+1)+'_'+str(n)+')', globals(), self.__dict__)
                        else:
                            exec('model_conv'+str(m+1)+'_'+str(n+1)+'=self.conv'+str(m+1)+'_'+str(n+1)+'(model_act'+str(m+1)+'_'+str(n)+')', globals(), self.__dict__)
                    exec('model_pad'+str(m+1)+'_'+str(n+1)+'=self.pad'+str(m+1)+'_'+str(n+1)+'(model_conv'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                    exec('model_act'+str(m+1)+'_'+str(n+1)+'=self.act'+str(m+1)+'_'+str(n+1)+'(model_pad'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                    if self.ifnormalization=='yes':
                        exec('model_norm'+str(m+1)+'_'+str(n+1)+'=self.norm'+str(m+1)+'_'+str(n+1)+'(model_act'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)   
                self.k=self.k+1
                self.l=self.l-1
            if if_last_act!='no':    
                if self.ifnormalization=='yes':
                    exec('model_convlast=self.convlast(model_norm'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                else:
                    exec('model_convlast=self.convlast(model_act'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                outputs=eval('self.actlast(model_convlast)', globals(), self.__dict__)
            else:
                if self.ifnormalization=='yes':
                    outputs=eval('self.convlast(model_norm'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
                else:
                    outputs=eval('self.convlast(model_act'+str(m+1)+'_'+str(n+1)+')', globals(), self.__dict__)
            return outputs
    if k_fold!=None: 
        models=[]
        for i in range(k_fold):
            models.append(Model(trainx[0:2],trainy[0:2],downnum,covnum,baselayer,cov_kernelsize,cov_strides,pool_method,pool_kernel_size,pool_strides,ifnormalization,normalization_method,activate))
    else:
        model=Model(trainx[0:2],trainy[0:2],downnum,covnum,baselayer,cov_kernelsize,cov_strides,pool_method,pool_kernel_size,pool_strides,ifnormalization,normalization_method,activate)
    if if_best_mode!='no':
        if k_fold!=None:
            for i in range(k_fold):
                models[i].load_state_dict(torch.load(modelpath+'_'+str(i+1)+'.pth', map_location=devices))
        else:
            model.load_state_dict(torch.load(modelpath+'.pth', map_location=devices))
    if k_fold!=None:
        for i in range(k_fold):
            models[i].to(devices)
    else:
        model.to(devices)
    if k_fold!=None:
        opts=[]
        for i in range(k_fold):
            if optimizer == 'SGD':
                opts.append(SGD(models[i].parameters(), lr=learning_rate))
            elif optimizer == 'Adam':
                opts.append(Adam(models[i].parameters(), lr=learning_rate))
            elif optimizer == 'Nadam':
                opts.append(NAdam(models[i].parameters(), lr=learning_rate))
    else:
        if optimizer == 'SGD':
            opt = SGD(model.parameters(), lr=learning_rate)
        elif optimizer == 'Adam':
            opt = Adam(model.parameters(), lr=learning_rate)
        elif optimizer == 'Nadam':
            opt = NAdam(model.parameters(), lr=learning_rate)
    if if_best_mode!='no':
        if k_fold!=None:
            for i in range(k_fold):
                opts[i].load_state_dict(torch.load(modelpath+'_'+str(i+1)+'_opt.pth', map_location=devices))
        else:
            opt.load_state_dict(torch.load(modelpath+'_opt.pth', map_location=devices))
    trainx=np.nan_to_num(trainx,nan=0)
    testx=np.nan_to_num(testx,nan=0)
    if if_print_model=='yes':
        if k_fold!=None:
            print(models[0])
        else:
            print(model)
    def _compute_shap_values_for_model_for_channel(model_instance, current_shap_data, x_sample_shape, output_channel_idx, devices):
        use_gradient_explainer = False 
        wrapped_model = ShapWrapper(model_instance, x_sample_shape, output_channel_idx=output_channel_idx).to(devices)
        wrapped_model.eval()

        background_data = current_shap_data
        
        if use_gradient_explainer:
            explainer = shap.GradientExplainer(wrapped_model, background_data)
        else:
            explainer = shap.DeepExplainer(wrapped_model, background_data)
        raw_shap_values = explainer.shap_values(current_shap_data)
        if isinstance(raw_shap_values, list):
            raw_shap_values = raw_shap_values[0]
        shap_x_reshaped = raw_shap_values.reshape(raw_shap_values.shape[0], *x_sample_shape)
        return np.abs(shap_x_reshaped)
    if ifrandom_split!='all_test':
        if valid_size!=None or k_fold !=None:
            if k_fold!=None:
                kf = KFold(n_splits=k_fold, shuffle=True, random_state=25)
                for fold_no, (train_idx, val_idx) in enumerate(kf.split(trainx, trainy)):
                    X_train_fold, y_train_fold = trainx[train_idx], trainy[train_idx]
                    X_val_fold, y_val_fold = trainx[val_idx], trainy[val_idx]
                    if fold_no==0:
                        train_loss=np.zeros((k_fold,int(np.ceil(X_train_fold.shape[0]/batch_size))))
                        train_metric=np.zeros((k_fold,int(np.ceil(X_train_fold.shape[0]/batch_size))))
                        test_loss=np.zeros((k_fold,int(np.ceil(X_val_fold.shape[0]/batch_size))))
                        test_metric=np.zeros((k_fold,int(np.ceil(X_val_fold.shape[0]/batch_size))))
                    if if_early_stopping!=None:
                        early_stopping = EarlyStopping(patience=if_early_stopping)
                    for i in range(epochs):
                        start = datetime.datetime.now()
                        for j in range(int(np.ceil(X_train_fold.shape[0]/batch_size))):
                            if j == int(X_train_fold.shape[0]/batch_size) :
                                train_output = models[fold_no](torch.tensor(X_train_fold[j*batch_size:],dtype=torch.float32,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.float32,device=devices)))
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:],dtype=torch.float32,device=devices)))
                            else:
                                train_output = models[fold_no](torch.tensor(X_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    train_losss = torch.mean(loss(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(y_train_fold[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                            train_loss[fold_no,j]=np.array(train_losss.item())
                            train_metric[fold_no,j]=np.array(train_metrics.item())
                            opts[fold_no].zero_grad()
                            train_losss.backward(retain_graph=True)
                            opts[fold_no].step()
                        with torch.no_grad():
                            for k in range(int(np.ceil(X_val_fold.shape[0]/batch_size))):
                                if k == int(X_val_fold.shape[0]/batch_size):
                                    test_output = models[fold_no](torch.tensor(X_val_fold[k*batch_size:],dtype=torch.float32,device=devices))
                                    if task_mode=='multi_classify':
                                        if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                            test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif loss_function=='NLLLoss':
                                            test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                            test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='CrossEntropyLoss':
                                            test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='NLLLoss':
                                            test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    else:
                                        test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.float32,device=devices)))
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:],dtype=torch.float32,device=devices)))
                                else:
                                    test_output = models[fold_no](torch.tensor(X_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices))
                                    if task_mode=='multi_classify':
                                        if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                            test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif loss_function=='NLLLoss':
                                            test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                            test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='CrossEntropyLoss':
                                            test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                        elif metrics=='NLLLoss':
                                            test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    else:
                                        test_losss= torch.mean(loss(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(y_val_fold[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                test_loss[fold_no,k]=np.array(test_losss.item())
                                test_metric[fold_no,k]=np.array(test_metrics.item())
                        if if_early_stopping!=None:
                            early_stopping(np.nanmean(test_loss[fold_no,:]), models[fold_no])
                            if early_stopping.early_stop:
                                early_stopping.load_best_checkpoint(models[fold_no])
                                break
                        end = datetime.datetime.now()
                        print('第',i+1,'次训练loss:',np.nanmean(train_loss[fold_no,:]),'，metric:',np.nanmean(train_metric[fold_no,:]),'  第',i+1,'次测试loss:',np.nanmean(test_loss[fold_no,:]),'，metric：',np.nanmean(test_metric[fold_no,:]),'训练用时:',end - start)
            else:
                if if_early_stopping!=None:
                    early_stopping = EarlyStopping(patience=if_early_stopping)
                for i in range(epochs):
                    start = datetime.datetime.now()
                    if i==0:
                        if ifrandom_split=='yes':
                            trainy,validy,trainx,validx = train_test_split(trainy,trainx,test_size=valid_size/(1-test_size),random_state=25)
                        else:
                            index=int((1-valid_size/(1-test_size))*trainy.shape[0])
                            validy=trainy[index:]
                            trainy=trainy[:index]
                            validx=trainx[index:]
                            trainx=trainx[:index]
                    train_loss=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                    train_metric=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                    test_loss=np.zeros((int(np.ceil(validx.shape[0]/batch_size))))
                    test_metric=np.zeros((int(np.ceil(validx.shape[0]/batch_size))))
                    for j in range(int(np.ceil(trainx.shape[0]/batch_size))):
                        if j == int(trainx.shape[0]/batch_size) :
                            train_output = model(torch.tensor(trainx[j*batch_size:],dtype=torch.float32,device=devices))
                            if task_mode=='multi_classify':
                                if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                    train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif loss_function=='NLLLoss':
                                    train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                    train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='CrossEntropyLoss':
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='NLLLoss':
                                    train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                            else:
                                train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                                train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                        else:
                            train_output = model(torch.tensor(trainx[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices))
                            if task_mode=='multi_classify':
                                if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                    train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif loss_function=='NLLLoss':
                                    train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                    train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='CrossEntropyLoss':
                                    train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='NLLLoss':
                                    train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                            else:
                                train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                                train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                        train_loss[j]=np.array(train_losss.item())
                        train_metric[j]=np.array(train_metrics.item())
                        opt.zero_grad()
                        train_losss.backward(retain_graph=True)
                        opt.step() 
                    with torch.no_grad():
                        for k in range(int(np.ceil(validx.shape[0]/batch_size))):
                            if k == int(validx.shape[0]/batch_size) :
                                test_output = model(torch.tensor(validx[k*batch_size:],dtype=torch.float32,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.float32,device=devices)))
                                    test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:],dtype=torch.float32,device=devices)))
                            else:
                                test_output = model(torch.tensor(validx[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices))
                                if task_mode=='multi_classify':
                                    if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                        test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif loss_function=='NLLLoss':
                                        test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                        test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='CrossEntropyLoss':
                                        test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                    elif metrics=='NLLLoss':
                                        test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                else:
                                    test_losss= torch.mean(loss(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                    test_metrics=torch.mean(metric(test_output,torch.tensor(validy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                            test_loss[k]=np.array(test_losss.item())
                            test_metric[k]=np.array(test_metrics.item())
                    if if_early_stopping!=None:
                        early_stopping(np.nanmean(test_loss), model)
                        if early_stopping.early_stop:
                            early_stopping.load_best_checkpoint(model)
                            break
                    end = datetime.datetime.now()
                    print('第',i+1,'次训练loss:',np.nanmean(train_loss),'，metric:',np.nanmean(train_metric),'  第',i+1,'次测试loss:',np.nanmean(test_loss),'，metric：',np.nanmean(test_metric),'训练用时:',end - start)
        else:
            if if_early_stopping!=None:
                early_stopping = EarlyStopping(patience=if_early_stopping)
            for i in range(epochs):
                start = datetime.datetime.now()
                train_loss=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                train_metric=np.zeros((int(np.ceil(trainx.shape[0]/batch_size))))
                test_loss=np.zeros((int(np.ceil(testx.shape[0]/batch_size))))
                test_metric=np.zeros((int(np.ceil(testx.shape[0]/batch_size))))
                for j in range(int(np.ceil(trainx.shape[0]/batch_size))):
                    if j == int(trainx.shape[0]/batch_size):
                        train_output = model(torch.tensor(trainx[j*batch_size:],dtype=torch.float32,device=devices))
                        if task_mode=='multi_classify':
                            if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                            elif loss_function=='NLLLoss':
                                train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                            if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                            elif metrics=='CrossEntropyLoss':
                                train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                            elif metrics=='NLLLoss':
                                train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                        else:
                            train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                            train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:],dtype=torch.float32,device=devices)))
                    else:
                        train_output = model(torch.tensor(trainx[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices))
                        if task_mode=='multi_classify':
                            if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                            elif loss_function=='NLLLoss':
                                train_losss = torch.mean(loss(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                            if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                train_metrics = torch.mean(metric(torch.argmax(train_output, dim=1), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                            elif metrics=='CrossEntropyLoss':
                                train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                            elif metrics=='NLLLoss':
                                train_metrics = torch.mean(metric(torch.log(train_output).contiguous(), torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                        else:
                            train_losss = torch.mean(loss(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                            train_metrics = torch.mean(metric(train_output, torch.tensor(trainy[j*batch_size:(j+1)*batch_size],dtype=torch.float32,device=devices)))
                    train_loss[j]=np.array(train_losss.item())
                    train_metric[j]=np.array(train_metrics.item())
                    opt.zero_grad()
                    train_losss.backward(retain_graph=True)
                    opt.step()
                with torch.no_grad():
                    for k in range(int(np.ceil(testx.shape[0]/batch_size))):
                        if k == int(testx.shape[0]/batch_size) :
                            test_output = model(torch.tensor(testx[k*batch_size:],dtype=torch.float32,device=devices))
                            if task_mode=='multi_classify':
                                if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                    test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif loss_function=='NLLLoss':
                                    test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                    test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='CrossEntropyLoss':
                                    test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='NLLLoss':
                                    test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:],dtype=torch.long,device=devices).squeeze(1)))
                            else:
                                test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.float32,device=devices)))
                                test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:],dtype=torch.float32,device=devices)))
                        else:
                            test_output = model(torch.tensor(testx[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices))
                            if task_mode=='multi_classify':
                                if loss_function=='default' or loss_function=='CrossEntropyLoss':
                                    test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif loss_function=='NLLLoss':
                                    test_losss= torch.mean(loss(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                if metrics=='default' or metrics=='accuracy' or metrics=='f1' or metrics=='recall' or  metrics=='precision':
                                    test_metrics=torch.mean(metric(torch.argmax(test_output, dim=1),torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='CrossEntropyLoss':
                                    test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                                elif metrics=='NLLLoss':
                                    test_metrics=torch.mean(metric(torch.log(test_output).contiguous(),torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.long,device=devices).squeeze(1)))
                            else:
                                test_losss= torch.mean(loss(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                                test_metrics=torch.mean(metric(test_output,torch.tensor(testy[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices)))
                        test_loss[k]=np.array(test_losss.item())
                        test_metric[k]=np.array(test_metrics.item())
                if if_early_stopping!=None:
                    early_stopping(np.nanmean(test_loss), model)
                    if early_stopping.early_stop:
                        early_stopping.load_best_checkpoint(model)
                        break
                end = datetime.datetime.now()
                print('第',i+1,'次训练loss:',np.nanmean(train_loss),'，metric:',np.nanmean(train_metric),'  第',i+1,'次测试loss:',np.nanmean(test_loss),'，metric：',np.nanmean(test_metric),'训练用时:',end - start)
    if k_fold!=None:
        if task_mode=='multi_classify':
            predicty=np.zeros((k_fold,testy.shape[0],int(np.max(vy))+1,testx.shape[2],testx.shape[3]))
        else:
            predicty=np.zeros((k_fold,testy.shape[0],testy.shape[1],testx.shape[2],testx.shape[3]))
    else:
        if task_mode=='multi_classify':
            predicty=np.zeros((testy.shape[0],int(np.max(vy))+1,testx.shape[2],testx.shape[3]))
        else:
            predicty=np.zeros((testy.shape[0],testy.shape[1],testx.shape[2],testx.shape[3]))
    if k_fold!=None:
        for i in range(k_fold):
            for k in range(int(np.ceil(testx.shape[0]/batch_size))):
                if k == int(testx.shape[0]/batch_size) :
                    predicty_batch = models[i](torch.tensor(testx[k*batch_size:],dtype=torch.float32,device=devices))
                    predicty[i,k*batch_size:]=predicty_batch.cpu().detach().numpy()
                else:
                    predicty_batch = models[i](torch.tensor(testx[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices))
                    predicty[i,k*batch_size:(k+1)*batch_size]=predicty_batch.cpu().detach().numpy()
        predicty=np.nanmean(predicty,axis=0)
    else:
        for k in range(int(np.ceil(testx.shape[0]/batch_size))):
            if k == int(testx.shape[0]/batch_size) :
                predicty_batch = model(torch.tensor(testx[k*batch_size:],dtype=torch.float32,device=devices))
                predicty[k*batch_size:]=predicty_batch.cpu().detach().numpy()
            else:
                predicty_batch = model(torch.tensor(testx[k*batch_size:(k+1)*batch_size],dtype=torch.float32,device=devices))
                predicty[k*batch_size:(k+1)*batch_size]=predicty_batch.cpu().detach().numpy()
    predicty = np.nan_to_num(predicty,nan=0)
    r=np.zeros((testx.shape[2],testx.shape[3],testy.shape[1]))
    p=np.zeros((testx.shape[2],testx.shape[3],testy.shape[1]))
    f1score=np.zeros((testx.shape[2],testx.shape[3],testy.shape[1]))
    accuracyscore=np.zeros((testx.shape[2],testx.shape[3],testy.shape[1]))
    recallscore=np.zeros((testx.shape[2],testx.shape[3],testy.shape[1]))
    precisionscore=np.zeros((testx.shape[2],testx.shape[3],testy.shape[1]))
    for i in range(testx.shape[2]):
        for j in range(testx.shape[3]):
            for k in range(testy.shape[1]):
                if task_mode=='regression':
                    r[i,j,k],p[i,j,k] = pearsonr(predicty[:,k,i,j],testy[:,k,i,j])
                    r[i,j,k]=np.nan_to_num(r[i,j,k],nan=0)
                elif task_mode=='binary_classify':
                    predicty[:,k,i,j]=[int(round(predicty[l,k,i,j],0)) for l in range(predicty.shape[0])]
                    if metrics=='Recall':
                        r[i,j,k]=recall_score(testy[:,k,i,j], predicty[:,k,i,j])
                    elif metrics=='Precision':
                        r[i,j,k]=precision_score(testy[:,k,i,j], predicty[:,k,i,j])
                    else:
                        r[i,j,k]=accuracy_score(testy[:,k,i,j], predicty[:,k,i,j])
                    p[i,j,k]=0
                    f1score[i,j,k]=f1_score(testy[:,k,i,j],predicty[:,k,i,j])
                    accuracyscore[i,j,k]=accuracy_score(testy[:,k,i,j],predicty[:,k,i,j])
                    recallscore[i,j,k]=recall_score(testy[:,k,i,j],predicty[:,k,i,j])
                    precisionscore[i,j,k]=precision_score(testy[:,k,i,j],predicty[:,k,i,j])
                elif task_mode=='multi_classify':
                    r[i,j,k]=accuracy_score(testy[:,k,i,j], np.argmax(predicty[:,:,i,j],axis=1))
                    p[i,j,k]=0
    if ifmute == 'no':
        if task_mode=='regression':
            print('相关系数',np.nanmean(r))
        elif task_mode=='binary_classify':
            print('召回率+精确率',np.nanmean(f1score),'准确率',np.nanmean(accuracyscore),'召回率',np.nanmean(recallscore),'精确率',np.nanmean(precisionscore))
        elif task_mode=='multi_classify':
            print('准确率',np.nanmean(r))
    should_run_shap_explainer = False
    if ifheatmap == 'yes' or ifweight in ['yes', 'shap']:
        should_run_shap_explainer = True

    if should_run_shap_explainer:
        current_testx_data = testx
        if testx.shape[0] >= 100:
            index = np.random.randint(0, testx.shape[0], size=100)
            current_testx_data = testx[index, ]

        x_sample_shape = current_testx_data.shape[1:] 

        flat_current_testx_tensor = torch.tensor(current_testx_data, dtype=torch.float32, device=devices).flatten(start_dim=1)
        
        if task_mode=='multi_classify':
            num_model_outputs = int(np.max(vy))+1
        else:
            num_model_outputs = testy.shape[1] # C_out

        shap_values_per_output_channel = []
        for output_ch_idx in range(num_model_outputs):
            if k_fold!=None:
                fold_shap_results_for_channel = []
                for model_k_fold in models:
                    model_k_fold.eval()
                    fold_shap_results_for_channel.append(_compute_shap_values_for_model_for_channel(
                        model_k_fold, flat_current_testx_tensor, x_sample_shape, output_ch_idx, devices
                    ))
                shap_values_per_output_channel.append(np.nanmean(np.stack(fold_shap_results_for_channel, axis=0), axis=0))
            else:
                model.eval()
                shap_values_per_output_channel.append(_compute_shap_values_for_model_for_channel(
                    model, flat_current_testx_tensor, x_sample_shape, output_ch_idx, devices
                ))
        
        shap_values_aggregated = np.stack(shap_values_per_output_channel, axis=0)

        if ifheatmap=='yes' and testx.ndim==4:
            heatmap_per_output_channel = np.nanmean(shap_values_aggregated, axis=1) 
            
            heatmap_output_list = []
            for i in range(num_model_outputs):
                current_heatmap_slice = heatmap_per_output_channel[i, :, :, :] 

                if len(x_sample_shape) == 3: 
                    current_heatmap_slice = current_heatmap_slice.transpose(1, 2, 0) 
                
                heatmap_output_list.append(current_heatmap_slice)
            
            if len(heatmap_output_list) > 0 and all(h.shape == heatmap_output_list[0].shape for h in heatmap_output_list):
                 heatmap = np.stack(heatmap_output_list, axis=0)
            else:
                 heatmap = heatmap_output_list 

        if ifweight in ['yes', 'shap']:
            weights_aggregated_per_channel_feature = np.nanmean(shap_values_aggregated, axis=(1, 3, 4))
            
            input_channel_for_weights = trainx.shape[1] 
            
            weights_final_output = np.zeros((num_model_outputs, input_channel_for_weights))

            for i in range(num_model_outputs): 
                current_output_channel_weights = weights_aggregated_per_channel_feature[i, :] 
                sum_for_norm = np.nansum(current_output_channel_weights) 
                
                if sum_for_norm != 0:
                    weights_final_output[i, :] = (current_output_channel_weights / sum_for_norm) * 100
                else:
                    weights_final_output[i, :] = 0.0 
                
                if ifmute == 'no': 
                    for j in range(input_channel_for_weights): 
                        print('预报因子',j+1,'对预报值',i+1,'的贡献：',np.array(weights_final_output[i,j]),'％')
                    print('\n')
            weights = weights_final_output
    elif ifweight == 'oob': 
        weights = np.zeros((testy.shape[1],testx.shape[1])) # C_out x C_in
        weight_more=np.zeros((testy.shape[1],testx.shape[1]))
        for i in tqdm(range(testy.shape[1])):
            for j in range(testx.shape[1]): 
                testx_new=copy.deepcopy(testx)
                weight_current_feature_output_pair = [] 
                for k in range(10): 
                    per=np.random.permutation(testx.shape[0])
                    testx_shuffle=testx[per,j,:,:] 
                    testx_new[:,j,:,:]=testx_shuffle 
                    
                    predicty_new_agg = None
                    if k_fold!=None:
                        predicty_new_folds = []
                        for o in range(k_fold):
                            predicty_new_batch_list = []
                            for n in range(int(np.ceil(testx.shape[0]/batch_size))):
                                if n == int(testx.shape[0]/batch_size):
                                    current_batch_x = torch.tensor(testx_new[n*batch_size:],dtype=torch.float32,device=devices)
                                else:
                                    current_batch_x = torch.tensor(testx_new[n*batch_size:(n+1)*batch_size],dtype=torch.float32,device=devices)
                                predicty_new_batch = models[o](current_batch_x)
                                predicty_new_batch_list.append(predicty_new_batch.cpu().detach().numpy())
                            predicty_new_folds.append(np.concatenate(predicty_new_batch_list, axis=0))
                        predicty_new_agg = np.nanmean(np.stack(predicty_new_folds, axis=0), axis=0)
                    else:
                        predicty_new_batch_list = []
                        for n in range(int(np.ceil(testx.shape[0]/batch_size))):
                            if n == int(testx.shape[0]/batch_size):
                                current_batch_x = torch.tensor(testx_new[n*batch_size:],dtype=torch.float32,device=devices)
                            else:
                                current_batch_x = torch.tensor(testx_new[n*batch_size:(n+1)*batch_size],dtype=torch.float32,device=devices)
                            predicty_new_batch = model(current_batch_x)
                            predicty_new_batch_list.append(predicty_new_batch.cpu().detach().numpy())
                        predicty_new_agg = np.concatenate(predicty_new_batch_list, axis=0)

                    for l in range(testy.shape[2]): 
                        for m in range(testy.shape[3]): 
                            if task_mode=='regression':
                                current_mse_original = sklearn.metrics.mean_squared_error(testy[:,i,l,m],predicty[:,i,l,m])
                                current_mse_shuffled = sklearn.metrics.mean_squared_error(testy[:,i,l,m],predicty_new_agg[:,i,l,m])
                                weight_current_feature_output_pair.append(current_mse_shuffled - current_mse_original)
                            elif task_mode=='multi_classify':
                                current_logloss_original = sklearn.metrics.log_loss(testy[:,i,l,m],predicty[:,:,l,m])
                                current_logloss_shuffled = sklearn.metrics.log_loss(testy[:,i,l,m],predicty_new_agg[:,:,l,m])
                                weight_current_feature_output_pair.append(current_logloss_shuffled - current_logloss_original)
                            else: 
                                current_logloss_original = sklearn.metrics.log_loss(testy[:,i,l,m],predicty[:,i,l,m])
                                current_logloss_shuffled = sklearn.metrics.log_loss(testy[:,i,l,m],predicty_new_agg[:,i,l,m])
                                weight_current_feature_output_pair.append(current_logloss_shuffled - current_logloss_original)
                
                weight_more[i,j]=np.nanmean(weight_current_feature_output_pair)

        for i in range(testy.shape[1]): 
            current_output_channel_weights_oob = weight_more[i,:] 
            sum_for_norm = np.nansum(current_output_channel_weights_oob)
            if sum_for_norm != 0:
                weights[i,:] = (current_output_channel_weights_oob / sum_for_norm) * 100
            else:
                weights[i,:] = 0.0

            if ifmute == 'no':
                for j in range(testx.shape[1]): 
                    print('预报因子',j+1,'对预报值',i+1,'的贡献：',np.array(weights[i,j]),'％')
                print('\n')
    if ifsave=='yes':
        if k_fold!=None:
            for i in range(k_fold):
                torch.save(models[i].state_dict(),savepath+'_'+str(i+1)+'.pth')
                torch.save(opts[i].state_dict(),savepath+'_'+str(i+1)+'_opt.pth')
        else:
            torch.save(model.state_dict(),savepath+'.pth')
            torch.save(opt.state_dict(),savepath+'_opt.pth')
    if k_fold!=None:    
        return models,predicty,testy,r,p,heatmap,weights
    else:
        return model,predicty,testy,r,p,heatmap,weights